# RFQ Conversion Prediction

This notebook builds a baseline model to estimate the probability that an incoming RFQ results in an executed trade.

The model uses synthetic historical RFQ data generated in the previous notebook.

The objective is to estimate trade conversion probability using client behaviour, product characteristics and current market conditions.

A possible extension would be to forecast directional client flow before an RFQ arrives. For example, identifying which clients are more likely to buy or sell under specific market conditions.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
historical_rfqs = pd.read_csv(
    "../data/synthetic_historical_rfqs.csv",
    parse_dates=["timestamp"]
)

historical_rfqs = (
    historical_rfqs
    .sort_values("timestamp")
    .reset_index(drop=True)
)

historical_rfqs.head()

,rfq_id,client,product_type,underlying,direction,notional,underlying_move_pct,iv_change,bid_ask_spread_pct,distance_to_barrier_pct,traded,timestamp
0,3130,Client32,Autocall,NVDA,SELL,1080000.0,2.031788,-0.594538,0.347405,1.955365,1,2025-01-01 08:03:00
1,4218,Client34,Autocall,NESN,BUY,380000.0,1.700107,3.292242,0.629700,11.808964,0,2025-01-01 08:03:00
2,1834,Client31,Autocall,TSLA,BUY,800000.0,1.455117,2.606704,0.338126,39.432096,1,2025-01-01 08:07:00
3,3278,Client18,Reverse Convertible,AAPL,BUY,850000.0,5.295314,3.887962,0.180865,NaN,1,2025-01-01 08:09:00
4,1105,Client37,Autocall,AAPL,SELL,290000.0,-2.833395,-0.971015,0.638075,25.495218,0,2025-01-01 08:11:00


## 1. Feature Preparation and Data Preprocessing

We prepare the variables that will be available when an RFQ arrives and define the target variable to predict.

The dataset contains both categorical and numerical variables. Categorical variables are converted into numerical indicators with one-hot encoding, while numerical variables are kept in interpretable units.

In [3]:
# Use the magnitude of market and volatility moves
historical_rfqs["abs_underlying_move_pct"] = historical_rfqs["underlying_move_pct"].abs()

historical_rfqs["abs_iv_change"] = historical_rfqs["iv_change"].abs()

# Indicates whether the product has a barrier
historical_rfqs["has_barrier"] = (
    historical_rfqs["distance_to_barrier_pct"].notna().astype(int)
)

# For products without a barrier, set distance to zero
historical_rfqs["distance_to_barrier_pct"] = (
    historical_rfqs["distance_to_barrier_pct"].fillna(0)
)

# make it easier to read
historical_rfqs["notional_mn"] = (historical_rfqs["notional"] / 1_000_000)


features = [
    "client",
    "product_type",
    "underlying",
    "direction",
    "notional_mn",
    "abs_underlying_move_pct",
    "abs_iv_change",
    "bid_ask_spread_pct",
    "distance_to_barrier_pct",
    "has_barrier"
]

X = historical_rfqs[features]
y = historical_rfqs["traded"]


X = pd.get_dummies(
    X,
    columns=["client", "product_type", "underlying", "direction"],
    dtype=int
)


# Use the first 80% of RFQs for training and the most recent 20% for testing

split_index = int(len(historical_rfqs)*0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]


## 2. Baseline Logistic Regression and Evaluation

We use logistic regression as an interpretable baseline model to estimate the probability that an incoming RFQ results in a trade.

The model is trained on the first 80% of the historical RFQs and evaluated on the most recent 20%, preserving the chronological order of the data. This simulates a realistic setting in which past client behaviour is used to predict future RFQ conversion.

Model performance is evaluated using the ROC-AUC score.

ROC-AUC measures how well the model ranks traded RFQs above non-traded RFQs across all possible classification thresholds. A score of 0.5 corresponds to random ranking, while a score of 1.0 represents perfect discrimination.

Unlike accuracy, ROC-AUC does not require choosing a fixed probability threshold and is therefore useful when the main objective is to rank RFQs by their likelihood of execution.

The baseline model achieves a ROC-AUC of 0.666, indicating that it captures meaningful but incomplete information about client trading behaviour.

In [4]:
model = LogisticRegression(max_iter=10000)

model.fit(
    X_train,
    y_train
)


coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
})
coefficients = coefficients.sort_values("coefficient",ascending=False)

pd.concat([
    coefficients.head(5),
    coefficients.tail(5)
])

,feature,coefficient
28,client_Client23,2.161846
12,client_Client07,1.270714
13,client_Client08,1.030078
33,client_Client28,0.806136
36,client_Client31,0.640196
17,client_Client12,-0.757330
9,client_Client04,-0.776581
25,client_Client20,-0.880261
45,client_Client40,-1.055820
8,client_Client03,-1.124991


In [5]:
trade_probabilities = model.predict_proba(X_test)[:, 1] #only second column = P(traded)


In [6]:
results = historical_rfqs.iloc[split_index:].copy()

results ["predicted_trade_probability"] = trade_probabilities

results[[
    "client",
    "product_type",
    "notional_mn",
    "traded",
    "predicted_trade_probability"
]].head(10)

,client,product_type,notional_mn,traded,predicted_trade_probability
4000,Client09,Autocall,0.30,0,0.255522
4001,Client34,Autocall,0.32,0,0.513020
4002,Client01,Autocall,0.11,0,0.372609
4003,Client22,Autocall,0.22,1,0.458031
4004,Client02,Autocall,2.04,1,0.282173
4005,Client28,Autocall,0.34,1,0.637063
4006,Client36,Barrier Reverse Convertible,0.77,1,0.448207
4007,Client13,Barrier Reverse Convertible,0.23,0,0.456239
4008,Client21,Autocall,0.11,0,0.365967
4009,Client15,Autocall,2.02,0,0.569363


In [7]:
roc_auc = roc_auc_score(
    y_test,
    trade_probabilities
)

print("ROC-AUC:", round(roc_auc, 3))

ROC-AUC: 0.666


## 3. Client-Product Interaction

The baseline model treats the client and product type as separate effects.

However, client behaviour may depend on the specific product being requested. A client who frequently trades Autocalls may have a very different conversion rate when requesting a Tracker.

We therefore add a client-product interaction feature so that the model can learn historical conversion patterns for specific client-product combinations.

In [8]:
# Create client-product interaction
historical_rfqs["client_product"] = (
    historical_rfqs["client"]
    + "_"
    + historical_rfqs["product_type"]
)

In [9]:
features = [
    "client",
    "product_type",
    "client_product",
    "underlying",
    "direction",
    "notional_mn",
    "abs_underlying_move_pct",
    "abs_iv_change",
    "bid_ask_spread_pct",
    "distance_to_barrier_pct",
    "has_barrier"
]

X_interaction = historical_rfqs[features]

X_interaction = pd.get_dummies(
    X_interaction,
    columns=[
        "client",
        "product_type",
        "client_product",
        "underlying",
        "direction"
    ],
    dtype=int
)

X_train_interaction = X_interaction.iloc[:split_index]
X_test_interaction = X_interaction.iloc[split_index:]

In [10]:
# second model
interaction_model = LogisticRegression(max_iter=10000)

interaction_model.fit(X_train_interaction,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [11]:
interaction_trade_probabilities = (
    interaction_model.predict_proba(X_test_interaction)[:, 1]
)

In [12]:
interaction_roc_auc = roc_auc_score(
    y_test,
    interaction_trade_probabilities
)

print("Baseline ROC-AUC:", round(roc_auc, 3))
print("Interaction ROC-AUC:", round(interaction_roc_auc, 3))

Baseline ROC-AUC: 0.666
Interaction ROC-AUC: 0.659


### Interaction Model Result

Adding the client-product interaction did not improve out-of-sample performance.

The baseline ROC-AUC was 0.666, compared with 0.659 for the interaction model.

Although client-product preferences are economically plausible, introducing many specific client-product combinations increases model complexity and reduces the number of observations available for each combination.

For this synthetic dataset, the additional interaction therefore does not improve generalization. 

## 4. Probability Calibration

ROC-AUC measures how well the model ranks RFQs, but it does not tell us whether the predicted probabilities themselves are reliable.

For a probability model used in a trading workflow, calibration is also important. If the model assigns approximately 60% trade probability to a group of RFQs, we would ideally observe a trade rate close to 60% within that group.

We therefore compare predicted probabilities with realised trade rates across probability buckets.

In [13]:
results["probability_group"] = pd.qcut(
    results["predicted_trade_probability"],
    q=5
)

In [14]:
calibration_table = (
    results
    .groupby("probability_group", observed=True)
    .agg(
        rfqs=("traded", "count"),
        predicted_probability=("predicted_trade_probability", "mean"),
        actual_trade_rate=("traded", "mean")
    )
)

calibration_table

,rfqs,predicted_probability,actual_trade_rate
probability_group,,,
"(0.129, 0.285]",200,0.228336,0.285
"(0.285, 0.371]",200,0.328517,0.300
"(0.371, 0.454]",200,0.412328,0.415
"(0.454, 0.547]",200,0.499623,0.455
"(0.547, 0.907]",200,0.655202,0.680


### Calibration Result

The predicted probabilities show reasonable calibration across the test set.

RFQs assigned lower trade probabilities exhibit lower realised conversion rates, while higher-probability RFQs show substantially higher realised trade rates.

For example, the lowest probability group has an average predicted probability of approximately 23% and a realised trade rate of 29%, while the highest group has a predicted probability of approximately 66% and a realised trade rate of 68%.

This suggests that, although the model's ranking ability is moderate (ROC-AUC = 0.666), its probability estimates provide useful information about expected RFQ conversion.

In [15]:
# Convert predicted probabilities into a binary prediction
results["predicted_trade"] = (
    results["predicted_trade_probability"] >= 0.50
).astype(int)

threshold_summary = (
    results
    .groupby("predicted_trade")
    .agg(
        rfqs=("traded", "count"),
        actual_trades=("traded", "sum"),
        actual_trade_rate=("traded", "mean")
    )
)

threshold_summary

,rfqs,actual_trades,actual_trade_rate
predicted_trade,,,
0,703,244,0.347084
1,297,183,0.616162


### 50% Classification Threshold

Using a 50% probability threshold, the model predicts 297 RFQs as likely to trade. Of these, 183 actually trade, corresponding to a 61.6% realised trade rate.

Among the 703 RFQs classified below the 50% threshold, 244 still result in a trade.

The model therefore provides useful separation between higher- and lower-conversion RFQs, but a fixed 50% threshold misses a substantial number of executed trades. This suggests that the predicted probabilities may be more useful for ranking and prioritisation than as a strict binary trade/no-trade decision.

## 5. RFQ Ranking Analysis

In practice, predicted trade probabilities may be more useful for ranking RFQs than for making a strict trade/no-trade classification.

We therefore evaluate the realised trade rate among the RFQs receiving the highest predicted probabilities.

In [16]:
ranked_results = results.sort_values(
    "predicted_trade_probability",
    ascending=False
)

for pct in [0.10, 0.20, 0.30]:

    n_rfqs = int(len(ranked_results) * pct)

    top_rfqs = ranked_results.head(n_rfqs)

    trade_rate = top_rfqs["traded"].mean()

    print(
        f"Top {int(pct * 100)}%: "
        f"{n_rfqs} RFQs, "
        f"trade rate = {trade_rate:.1%}"
    )

Top 10%: 100 RFQs, trade rate = 75.0%
Top 20%: 200 RFQs, trade rate = 68.0%
Top 30%: 300 RFQs, trade rate = 61.7%


In [17]:
overall_trade_rate = results["traded"].mean()

print(
    f"Overall test trade rate: "
    f"{overall_trade_rate:.1%}"
)

Overall test trade rate: 42.7%


### Ranking Result

The ranking analysis shows that the model successfully concentrates executed trades among the highest-ranked RFQs.

While the overall trade rate in the test set is 42.7%, the realised trade rate increases to:

- 61.3% for the top 30% of RFQs,
- 68.0% for the top 20%,
- 75.0% for the top 10%.

This suggests that the model is more useful as a prioritisation tool than as a strict binary trade/no-trade classifier.

In a trading workflow, the predicted probability could therefore help traders identify which incoming RFQs are more likely to convert and where faster attention may have greater value.

## 6. Historical Client Hit Rate

Client identity captures differences in trading behaviour, but a more directly interpretable measure is the client's historical RFQ conversion rate.

For each RFQ, we calculate the client's hit rate using only RFQs that occurred before the current one.

This ensures that the model only uses information that would have been available at the time the RFQ arrived and avoids target leakage.

For RFQs in the test set, client hit rates are calculated using the historical information available from the training period.

In [18]:
train_rfqs = historical_rfqs.iloc[:split_index].copy()
test_rfqs = historical_rfqs.iloc[split_index:].copy()

In [ ]:
# Calculate each client's historical trade rate using only previous RFQs

train_rfqs["client_hit_rate"] = (
    train_rfqs
    .groupby("client")["traded"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

# first RQF will be NaN (no previous history)
train_rfqs["client_hit_rate"] = (
    train_rfqs["client_hit_rate"]
    .fillna(0.50)
)

In [20]:
# value to assign in the test set
client_hit_rates = (train_rfqs.groupby("client")["traded"].mean())

test_rfqs["client_hit_rate"] = (
    test_rfqs["client"].map(client_hit_rates)
)

# if we have a client showing up the first time
test_rfqs["client_hit_rate"] = (
    test_rfqs["client_hit_rate"].fillna(y_train.mean())
)

In [21]:
#add values to the model

X_train_hit_rate = X_train.copy()
X_test_hit_rate = X_test.copy()

X_train_hit_rate["client_hit_rate"] = (
    train_rfqs["client_hit_rate"].values
)

X_test_hit_rate["client_hit_rate"] = (
    test_rfqs["client_hit_rate"].values
)

hit_rate_model = LogisticRegression(
    max_iter=1000
)

hit_rate_model.fit(
    X_train_hit_rate,
    y_train
)

hit_rate_probabilities = (
    hit_rate_model.predict_proba(X_test_hit_rate)[:, 1]
)

hit_rate_roc_auc = roc_auc_score(
    y_test,
    hit_rate_probabilities
)

print("Baseline ROC-AUC:", round(roc_auc, 3))
print("Hit Rate ROC-AUC:", round(hit_rate_roc_auc, 3))

Baseline ROC-AUC: 0.666
Hit Rate ROC-AUC: 0.664


In [22]:
hit_rate_results = results.copy()

hit_rate_results["predicted_trade_probability"] = (
    hit_rate_probabilities
)

hit_rate_results = hit_rate_results.sort_values(
    "predicted_trade_probability",
    ascending=False
)

for pct in [0.10, 0.20, 0.30]:

    n_rfqs = int(len(hit_rate_results) * pct)

    trade_rate = (
        hit_rate_results
        .head(n_rfqs)["traded"]
        .mean()
    )

    print(
        f"Top {int(pct * 100)}%: "
        f"trade rate = {trade_rate:.1%}"
    )

Top 10%: trade rate = 74.0%
Top 20%: trade rate = 68.5%
Top 30%: trade rate = 62.3%


### Historical Hit Rate Result

Adding the historical client hit rate does not improve the baseline model.

ROC-AUC changes from 0.666 to 0.664, while ranking performance remains very similar.

This suggests that client identity already captures much of the information contained in the historical hit rate. The simpler baseline model is therefore retained.

## Conclusion

The baseline logistic regression captures a useful ranking signal in the synthetic RFQ data.

It achieves a ROC-AUC of 0.666. The overall test trade rate is 42.7%, compared with:

- 61.3% for the top 30% of ranked RFQs
- 68.0% for the top 20%
- 75.0% for the top 10%

Client-product interactions and historical client hit rates were also tested, but did not improve out-of-sample performance.

The baseline model is therefore retained as a simple way to rank incoming RFQs by expected conversion probability.